# 01. Sorted search

**Tier:** Deep. **Why you'd use this:** A phone book is sorted by name. You do not open every page; you open the middle, decide which half can still hold the name, and repeat until the page is found or the range is empty.

**After this lesson you can:** keep a closed index range that may hold the target; drop half the remaining indexes on each comparison.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What must be true of a list before binary search is correct?
2. If `nums[mid]` is smaller than the target, which half can you discard?

<details><summary>Answers</summary>

The list must be sorted in the order you compare. Discard the left half, including `mid`, by setting `left = mid + 1`. Skim [Lists](../../course/04-collections/01-lists-lesson.ipynb) and [Loops](../../course/03-control-flow/02-loops-lesson.ipynb) if either feels rusty.
</details>


## Recognize it

You have a sorted list of distinct numbers and need to know whether a target is present. The prompt talks about "search in a sorted array", "membership in log time", or "halve the range each step".

Signals: sorted input, ask about one value's presence or position, and a clear "too small / too large" comparison at the middle. A look-alike that is not this pattern: scanning an unsorted list — sorting first costs more than a single linear pass when you only ask once.


## Why it works this way

A linear scan compares the target to every value until it hits or finishes. On a miss at the end, that is `n` comparisons. Binary search starts from a different contract: because the list is sorted ascending, every value left of an index is less than or equal to that index's value, and every value to the right is greater. Comparing the middle value to the target therefore classifies a whole half of the remaining range as impossible.

The loop keeps two indexes, `left` and `right`, that bound a closed interval of candidates. The invariant is: if the target exists anywhere in the original list, it still lies between `left` and `right` inclusive. When `nums[mid] < target`, nothing at `mid` or to its left can be the answer, so `left` becomes `mid + 1`. When `nums[mid] > target`, nothing at `mid` or to its right can be the answer, so `right` becomes `mid - 1`. Equality returns immediately. When `left` passes `right`, the interval is empty and the target is absent.

Each step roughly halves the interval. After about `log₂ n` steps the interval collapses. Extra memory is two indexes plus the middle — constant. The cost of being wrong about sortedness is silent incorrect answers, not a crash: the algorithm still runs, but the discarded half may have held the target. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why random access at `mid` is cheap on a contiguous buffer; a linked structure would make the middle jump expensive and erase the win.

Returning a boolean is the membership sibling. Returning an insert index is the same loop with a different stopping rule: keep searching for the first value that is not smaller than the target, then return that boundary. Both rest on the same half-discard invariant.


## When to use it

Use binary search when the list is already sorted and you will ask about membership or position often enough that `log n` beats a scan. Prefer a plain loop over an unsorted list when sorting would dominate or you ask only once. Prefer a hash set when you need average constant membership and can afford the memory and unordered storage. Prefer two pointers on a sorted list when the question is about a pair of values, not a single target.


## Brute force, and why it's slow

Count comparisons. Walk left to right until the target appears or the list ends.


In [1]:
def contains_slow(nums, target):
    comparisons = 0
    for value in nums:
        comparisons += 1
        if value == target:
            return True, comparisons
    return False, comparisons


for size in (50, 200, 800):
    data = list(range(size))
    print(size, contains_slow(data, size - 1))
    print(size, contains_slow(data, -1))


50 (True, 50)
50 (False, 50)
200 (True, 200)
200 (False, 200)
800 (True, 800)
800 (False, 800)


## Key insight

While `left <= right`, the middle index splits a sorted range. One comparison throws away the half that cannot hold the target. Stop when the value matches or the range is empty.


## Worked example

Return whether `target` appears in a sorted list of distinct integers. Empty list means absent.


In [2]:
def contains(nums: list[int], target: int) -> bool:
    # 1. Closed range of indexes that may still hold the target.
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        # 2. Hit: done.
        if nums[mid] == target:
            return True
        # 3. Drop the half that cannot hold it.
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return False


print(contains([1, 3, 5, 7], 5))
print(contains([1, 3, 5, 7], 2))
print(contains([], 1))


True
False
False


## Trace

`[1, 3, 5, 7]`, target `5`:

| left | right | mid | nums[mid] | action |
|---|---|---|---|---|
| 0 | 3 | 1 | 3 | 3 < 5 → left = 2 |
| 2 | 3 | 2 | 5 | equal → **True** |

Target `2` on the same list ends with `left = 1`, `right = 0` after discarding both sides of `3`, then returns `False`.


## Cost

Each step halves the remaining indexes, so time grows with `log n`. Extra memory is constant. The linear scan pays up to `n` comparisons on a miss; binary search pays a handful even for hundreds of values.


## Variants

Not solved here: [Binary Search](../../challenges/problems/09-binary-search.ipynb), [Search in Rotated Sorted Array](../../challenges/problems/10-search-in-rotated-sorted-array.ipynb), [Find Minimum in Rotated Sorted Array](../../challenges/problems/11-find-minimum-in-rotated-sorted-array.ipynb).


## Common mistake

This bug is silent. The loop uses `left < right` and never tests the final remaining index when the range shrinks to one cell. There is no traceback. The print is `False`, and the answer is `True`.


In [3]:
def contains_wrong(nums, target):
    left, right = 0, len(nums) - 1
    while left < right:  # skips the last remaining index
        mid = (left + right) // 2
        if nums[mid] == target:
            return True
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return False


print(contains_wrong([1, 3, 5], 5))


False


The fix keeps `left <= right` so a one-element range is still checked. Run `contains` on `[1, 3, 5]` with target `5` and you get `True`.


In [4]:
print(contains([1, 3, 5], 5))


True


## Python idioms

`bisect.bisect_left` returns the insertion index in a sorted list. Membership is then a bounds check plus an equality test at that index. The hand-written loop makes the invariant obvious; the library call is what you reach for in production once the idea is clear.


In [5]:
import bisect


def contains_lib(nums: list[int], target: int) -> bool:
    index = bisect.bisect_left(nums, target)
    return index < len(nums) and nums[index] == target


print(contains_lib([1, 3, 5, 7], 5))
print(contains_lib([1, 3, 5, 7], 2))


True
False


## Recap

- Sorted order lets one middle comparison discard half the range.
- Keep a closed `[left, right]` until it empties or hits.
- `left <= right` still tests the last remaining index.

Next: [practice](01-search-insert-practice.ipynb).
